# 📓 Semana 1 · Dia 4 — SQL Warehouse, queries analíticas e views

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (SQL analítico) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | 10 queries analíticas respondidas + ponte Python↔SQL |

---


## 📖 Teoria — SQL Warehouse (e por que a Free Edition tem 1)

O **SQL Warehouse** é o compute especializado em SQL (otimizado para BI, dashboards e queries ad hoc). Na Free Edition você tem **1 warehouse de tamanho 2X-Small** — suficiente para o curso. Ele pode ser iniciado/parado manualmente; na Free, ele desliga sozinho após inatividade.

**Diferença**: notebooks rodam no compute serverless de notebooks; queries SQL e dashboards rodam no SQL Warehouse. Ambos acessam o **mesmo Unity Catalog**.


### 💻 Na prática — Conectando no SQL Warehouse

1. Em **Compute → SQL Warehouses**, crie um warehouse (2X-Small, auto-stop ~10 min).
2. Abra **Queries** (ícone de banco de dados) para escrever SQL puro.
3. Alternativamente, use células `%sql` no notebook — elas também rodam via Spark, mas para testar o warehouse de verdade, use o editor de Queries.


## 📖 Teoria — Views temporárias — a ponte Python ↔ SQL

Uma **view temporária** registra um DataFrame para ser consultado em SQL na mesma sessão. É a peça que conecta seu código Python com a consulta SQL — muito usado em pipelines e em perguntas de prova.

```
df.createOrReplaceTempView('nome')
spark.sql('SELECT ... FROM nome')
```


In [ ]:
# Ponte Python → SQL: registrar view e consultar
df_vendas = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/bronze/vol_dados_curso/vendas.csv")
df_vendas.createOrReplaceTempView("vendas_vw")
print("View vendas_vw registrada. Total de linhas:", spark.sql("SELECT COUNT(*) FROM vendas_vw").collect()[0][0])

In [ ]:
%sql
-- 1. Vendas por país (análise clássica)
SELECT Country, COUNT(*) AS vendas
FROM vendas_vw
GROUP BY Country
ORDER BY vendas DESC
LIMIT 10

In [ ]:
# SQL → Python: trazer o resultado de volta
resultado = spark.sql("""
    SELECT SUM(Quantity * UnitPrice) AS receita_total
    FROM vendas_vw
""").collect()[0]["receita_total"]
print(f"Receita total do dataset: {resultado:,.2f}")

### 💻 Na prática — Queries analíticas essenciais

Rode e analise cada uma. Esse padrão (GROUP BY + agregação + ORDER BY + LIMIT) é o coração de 90% dos dashboards.


In [ ]:
%sql
-- 2. Ticket médio por país
SELECT Country, ROUND(AVG(Quantity * UnitPrice), 2) AS ticket_medio
FROM vendas_vw
GROUP BY Country
ORDER BY ticket_medio DESC
LIMIT 10

In [ ]:
%sql
-- 3. Receita por mês (extração de data com DATE_TRUNC)
SELECT DATE_TRUNC("month", InvoiceDate) AS mes, ROUND(SUM(Quantity * UnitPrice), 2) AS receita
FROM vendas_vw
GROUP BY mes
ORDER BY mes
LIMIT 12

> 🎯 **Dica de prova**: A prova DEA 2026 enfatiza **ELT com Spark SQL e Python**. Saber escrever agregações, CTEs, window functions e conectar Python↔SQL via views é o núcleo do domínio 1 da prova.


## 🎯 Exercícios de fixação

**1.** Qual a receita total do Reino Unido (United Kingdom)?

**2.** Quantos clientes únicos (CustomerID não nulo) existem?

**3.** Qual o dia com mais vendas do dataset?

**4.** Crie uma view `top_paises_vw` com os 10 países por receita e consulte-a em Python.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Receita UK

`SELECT ROUND(SUM(Quantity*UnitPrice),2) FROM vendas_vw WHERE Country='United Kingdom'` — resultado ~ R$ 7,5 milhões (o dataset é dominado pelo UK).

**2.** Clientes únicos

`SELECT COUNT(DISTINCT CustomerID) FROM vendas_vw WHERE CustomerID IS NOT NULL` — ~4.372.

**3.** Dia com mais vendas

`SELECT DATE_TRUNC('day', InvoiceDate) dia, COUNT(*) n FROM vendas_vw GROUP BY dia ORDER BY n DESC LIMIT 1` — tipicamente um dia de novembro (Black Friday).

**4.** View + Python

`spark.sql('CREATE OR REPLACE TEMP VIEW top_paises_vw AS SELECT Country, SUM(Quantity*UnitPrice) receita FROM vendas_vw GROUP BY Country ORDER BY receita DESC LIMIT 10')` e depois `spark.table('top_paises_vw').collect()`.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*